# SHACL Shapes — User Guide

`starshacl` extends pySHACL in line with the SHACL 1.2 draft specifications: validation over RDF 1.2 graphs (triple terms, direction-tagged literals), SHACL rules, fine-grained per-constraint severity, and SHACL 1.2 Node Expressions. This guide is a hub — it covers `StarShaclValidator`'s four processing modes and the validation-time features that don't have their own deep-dive, and links out to a dedicated guide for each topic substantial enough to need one:

- **[SHACL node expressions](04a-shacl-node-expressions.ipynb)** — the `shnex:`/`sparql:` node-expression vocabulary, `sh:values`, custom functions.
- **[SHACL inference rules](04b-shacl-inference-rules.ipynb)** — `sh:rule`/`sh:TripleRule`/`sh:SPARQLRule`, execution ordering, rule sets, provenance.
- **SHACL and SPARQL** (4.c) — `sh:sparql` constraints, user-defined `sh:ConstraintComponent`. 🚧 *not yet written.*
- **SHACL UI** (4.d) — `shui:` presentation metadata. 🚧 *not yet written.*
- **SHACL profiling** (4.e) — `sh:ShapesGraph`/`sh:DataGraph`/`owl:imports` packaging conventions. 🚧 *not yet written.*
- **[SHACL subgraph extraction & hashing](04f-shacl-subgraph-extraction.ipynb)** — `extract_subgraph()`, `close_shape()`, and the hash-and-verify commitment pattern.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later sections reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

## 1. What starshacl adds over pySHACL

- Validation over RDF 1.2 graphs — triple terms, reification, direction-tagged literals
- SHACL rule support (`sh:rule`, `sh:TripleRule`, `sh:SPARQLRule`, rule sets)
- Direction-aware datatype constraints
- Fine-grained, per-constraint severity and messages, including new severity levels (`sh:Debug`, `sh:Trace`)
- SHACL 1.2 Node Expressions (`shnex:` combinators, `sparql:`-namespaced SPARQL built-ins, and user-defined custom node expression functions)
- A fourth processing mode, `extract_subgraph()`, alongside pySHACL's own validate/apply-rules pair

## 2. Four processing modes

`StarShaclValidator` exposes four independent processing modes over the same data/shapes graph pair.

### 2.1 `validate()` — conformance checking, never mutates

The familiar SHACL entry point: given a data graph and a shapes graph, report whether the data conforms. The rest of this guide's examples (sections 3-4 below) are all `validate()` examples, showing RDF-1.2-specific and SHACL-1.2-specific behavior.

### 2.2 `apply_rules()` — executes `sh:rule`, materializes real triples

Runs every `sh:rule` in the shapes graph to a fixed point and returns the result merged into a *new* data graph — the original `data_graph` is never mutated. Full depth (execution ordering, rule sets, condition filtering, provenance) is in the [inference rules guide](04b-shacl-inference-rules.ipynb); here's the shortest possible example: a rule that adds `ex:trusted ex:yes` to any node reified with `ex:confidence "high"`.

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:confidence "high" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ConfidenceRule a sh:NodeShape ;
      sh:targetSubjectsOf ex:confidence ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject sh:this ;
        sh:predicate ex:trusted ;
        sh:object ex:yes ;
        sh:condition [ sh:property [ sh:path ex:confidence ; sh:hasValue "high" ] ] ;
      ] .
""", format="turtle")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print("new triple materialized:", (EX.claim, EX.trusted, EX.yes) in result.data_graph)
print("conforms:", result.conforms)

new triple materialized: True
conforms: True


### 2.3 `evaluate()` — computed properties for display, without storing them

A third, independent mode: compute every `sh:values`-declared *virtual* property and return it merged into a throwaway copy of the data graph, without ever writing to the real one. Useful for "display or query this as if it were stored" without actually persisting a derived value. Full depth (including what happens when a stored value already exists for the same property) is in the [node expressions guide](04a-shacl-node-expressions.ipynb#3.-Computed-properties:-sh:values); here, `ex:friendCount` is computed on demand rather than stored.

In [3]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:friendCount ;
                    sh:values [ shnex:count [ shnex:pathValues ex:friend ] ] ] .
""", format="turtle12")

display_result = StarShaclValidator().evaluate(data_graph=data, shacl_graph=shapes)
print("friend count:", [v.toPython() for v in display_result.data_graph.objects(EX.alice, EX.friendCount)])
print("original data_graph still has zero ex:friendCount triples:",
      len(list(data.triples((None, EX.friendCount, None)))) == 0)

friend count: [2]
original data_graph still has zero ex:friendCount triples: True


### 2.4 `extract_subgraph()` — pull exactly the subgraph a shape covers

Given a focus node that conforms to a shape, extracts exactly the subgraph of *real, stored* triples that shape's constraints actually covered — not the whole graph around it. The motivating use case (hash a commitment, later re-verify it hasn't changed) and the full mechanics are in the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb).

## 3. SHACL 1.2 node kinds and severities

### 3.1 Is a value a triple term? `sh:nodeKind ( sh:TripleTerm )`

The list-valued form of `sh:nodeKind` recognizes `sh:TripleTerm` as one of its choices, letting a shape require that a value — e.g. the object of `rdf:reifies` — is genuinely an RDF 1.2 triple term, not a plain URI, blank node, or literal. Only the list form works; a bare `sh:nodeKind sh:TripleTerm` (no list) silently falls through to plain pySHACL's own check, which has no concept of triple terms at all.

In [4]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    [] a sh:NodeShape ;
      sh:targetSubjectsOf rdf:reifies ;
      sh:property [ sh:path rdf:reifies ; sh:nodeKind ( sh:TripleTerm ) ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .
""", format="turtle12")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms (real triple term):", result.conforms)

bad_data = StarLayerGraph()
bad_data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies ex:not_a_triple_term .
""", format="turtle")
result_bad = StarShaclValidator().validate(data_graph=bad_data, shacl_graph=shapes, meta_shacl=False)
print("conforms (plain URI, not a triple term):", result_bad.conforms)

conforms (real triple term): True
conforms (plain URI, not a triple term): False


### 3.2 New severity levels: `sh:Debug`, `sh:Trace`

SHACL 1.2 adds two severities below `sh:Info`/`sh:Warning`. Unlike those two (which need `allow_warnings=True`/`allow_infos=True` to stop blocking `conforms`), a `sh:Debug`/`sh:Trace` result **never** blocks conformance — it's recorded in the report but has no effect on `result.conforms`, useful for constraints you want visibility into without failing validation over.

In [5]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:severity sh:Debug ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print("conforms despite missing ex:age:", result.conforms)  # True - sh:Debug never blocks
print(result.report_text)

conforms despite missing ex:age: True
Validation Report
Conforms: False
Results (1):
Validation Result in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Debug
	Source Shape: [ sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:age ; sh:severity sh:Debug ]
	Focus Node: ex:alice
	Result Path: ex:age
	Message: Less than 1 values on ex:alice->ex:age



### 3.3 Fine-grained severity via reification: `{| sh:severity ... |}`

A single constraint-value triple can carry its own `sh:severity`/`sh:deactivated` override via RDF 1.2's inline annotation shorthand — distinct from, and finer-grained than, a shape's own `sh:severity`. Currently wired for `sh:datatype`/`sh:uniqueMembers`/`sh:reificationRequired`/`sh:singleLine` (severity) and `sh:property` (deactivation), and only on a constraint declared directly on the targeted shape (not yet inside a nested `sh:property [...]` blank node).

In [6]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:AgeMustBeIntShape a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:datatype xsd:integer {| sh:severity sh:Warning |} .
""", format="turtle12")
data = StarLayerGraph()
data.add((EX.alice, EX.dummy, EX.alice))  # ex:alice is a URI, not an xsd:integer literal - violates

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
severities = {result.report_graph.qname(o) for _, _, o in
              result.report_graph.triples((None, SH.resultSeverity, None))}
print("severity from the annotation:", severities)  # {'sh:Warning'}, not the default sh:Violation
print("conforms:", result.conforms)                  # still False - Warning still blocks unless allowed

result2 = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False, allow_warnings=True)
print("conforms with allow_warnings=True:", result2.conforms)

severity from the annotation: {'sh:Warning'}
conforms: False
conforms with allow_warnings=True: True


### 3.4 Direction-aware datatype constraints

`rdf:dirLangString` (the datatype `DirLangString` literals carry) works as an ordinary `sh:datatype` value.

In [7]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:n1 a ex:Note ;
      ex:label "hello"@en--ltr .
""", format="turtle12")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:LangShape a sh:NodeShape ;
      sh:targetClass ex:Note ;
      sh:property [ sh:path ex:label ; sh:datatype rdf:dirLangString ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print("conforms:", result.conforms)

conforms: True


## 4. Node expressions

SHACL 1.2 Node Expressions — the `shnex:` combinator vocabulary, the `sparql:` namespace exposing SPARQL built-ins, and user-defined custom node expression functions (including "named node expressions") — have their own dedicated guide: **[SHACL node expressions](04a-shacl-node-expressions.ipynb)**.

## Not covered yet

The project tracks this explicitly against the live W3C drafts in `packages/shacl/docs/shacl12-gap-matrix.md`. As of this guide's last review:

- **Content-level validation of `sh:select`/`sh:ask`/`sh:construct` query *text*** — checked structurally today (cardinality, datatype), not parsed as real SPARQL. A deliberate scope decision, not pursued.
- **SHACL 1.2 UI's `shui:WidgetScore`/`shui:WidgetAcceptMatcher` widget-selection algorithm** — `shui:` annotations on shapes validate fine (see 4.d); the selection procedure itself is a genuinely new, optional feature, not built.
- **Calling a custom node expression function as an ordinary SPARQL function by name** from `sh:select`/`sh:sparqlExpr` query text (e.g. `ex:instanceCount(ex:Name)`) — the *node-expression* call forms of custom functions are supported today (see 4.a); this query-text form is a separate, larger mechanism (registering the function as a real SPARQL engine function).
- **Deferred**: SPARQL Rule Language (SRL) and SHACL's own Compact Syntax — see [3.a](03a-sparql-rules-pending.md).
- **Deferred**: `sh:PropertyRule` (Core's `sh:values`-based rule shorthand) — a genuine gap in pySHACL itself, not just untested here; using it raises `RuleLoadError`.

Already done, despite being listed as "planned" in earlier drafts of this guide: `sh:RuleSet`/`sh:hasRule`/`sh:includesRuleSet` (named rule subsets — `apply_rules(..., rule_set=<IRI>)`), `sh:resultAnnotation`/`sh:annotationProperty`/`sh:annotationVarName`/`sh:annotationValue` (for plain `sh:sparql` constraints), and SHACL 1.2 Profiling (investigated in full — no validator runtime behavior gap exists; see 4.e).